# Extraction of CMS Fraud Types and Detection Methods

This notebook reviews all 10 CMS fraud-search pages:

```text
https://www.cms.gov/search/cms?keys=fraud&page=0
...
https://www.cms.gov/search/cms?keys=fraud&page=9
```

The notebook extracts info from these websites and summarizes the information into :

1. **Fraud categories and fraud types**
2. **Fraud detection methods and technologies described by CMS**

It removes prevention, reporting, synthetic data, model training, and other unrelated content.

## Workflow


```text
Goes over 10 CMS search pages
→ collect and deduplicate CMS result links
→ read webpages and PDFs
→ summarize each CMS source
→ merge duplicate fraud labels
→ create fraud-category and detection-technology tables
→ saves information into Markdown, CSV, and JSON outputs
```

## 1. Setup

Install packages using:

```python
%pip install requests beautifulsoup4 pandas openai python-dotenv pypdf tabulate truststore
```

Then, create a `.env` file beside the notebook:

```text
OPENAI_API_KEY=your_key_here
OPENAI_MODEL=gpt-4o-mini
```

In [18]:
#if missing packages, uncomment the following line
# %pip install -q requests beautifulsoup4 pandas openai python-dotenv pypdf tabulate truststore

import io
import os
import re
import json
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, Markdown



load_dotenv()

SEARCH_URLS = [
    f"https://www.cms.gov/search/cms?keys=fraud&page={page}"
    for page in range(10)
]

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
MAX_RESULTS_PER_PAGE = 10
MAX_SOURCE_CHARACTERS = 25_000
TIMEOUT = 45
REQUEST_DELAY = 0.3

api_key =os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Add OPENAI_API_KEY to a .env file.")

client=OpenAI(api_key=api_key)



session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; CMSFraudResearch/1.0)"
})

print("CMS search pages:", len(SEARCH_URLS))
print("Model:",MODEL)

CMS search pages: 10
Model: gpt-4o-mini


## 2. Collect result links from pages 0–9

The code collects links only from the CMS search results section. Duplicate URLs are removed across all 10 search pages.

In [19]:
def normalize_cms_url(base_url, href):
    """Convert a relative link into a clean CMS URL."""
    url = urljoin(base_url, href).split("#", 1)[0]
    parsed = urlparse(url)

    if parsed.scheme not in {"http", "https"}:
        return None

    if parsed.netloc.lower() not in {"cms.gov", "www.cms.gov"}:
        return None

    return url


def get_search_page_results(search_url, max_results=10):
    """Collect result links from one CMS search page."""
    response = session.get(search_url, timeout=TIMEOUT)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    heading = soup.find(
        lambda tag: tag.name in {"h1", "h2"}
        and tag.get_text(" ", strip=True).lower() == "search results"
    )

    if heading is None:
        raise RuntimeError(f"Search results heading not found: {search_url}")

    results = []
    seen_urls = set()

    for element in heading.find_all_next():
        text = element.get_text(" ", strip=True)

        if element.name in {"h2", "h3", "nav"} and "pagination" in text.lower():
            break

        if element.name != "a" or not element.get("href"):
            continue

        title = text.strip()
        url = normalize_cms_url(search_url, element["href"])

        if not url or url == search_url:
            continue

        if len(title)<10 or url in seen_urls:
            continue

        results.append({
            "title": title,
            "url": url,
            "search_page": search_url,
        })
        seen_urls.add(url)

        if len(results)>=max_results:
            break

    return results


all_results = []

for page_number, search_url in enumerate(SEARCH_URLS):
    print(f"Collecting search page {page_number}...")
    all_results.extend(
        get_search_page_results(
            search_url,
            max_results=MAX_RESULTS_PER_PAGE,
        )
    )
    time.sleep(REQUEST_DELAY)

# Deduplicate across all 10 search pages.
result_map = {}

for item in all_results:
    result_map.setdefault(item["url"], item)

search_results = list(result_map.values())
results_df = pd.DataFrame(search_results)

print("Unique CMS result links:", len(results_df))
display(results_df[["title","url", "search_page"]])




if results_df.empty:
    raise RuntimeError("No CMS result links were collected.")

Unique CMS result links: 90


,title,url,search_page
0,About searching,https://www.cms.gov/search/cms/help,https://www.cms.gov/search/cms?keys=fraud&page=0
1,CMS Announces Aggressive Nationwide Crackdown ...,https://www.cms.gov/newsroom/press-releases/cm...,https://www.cms.gov/search/cms?keys=fraud&page=0
2,Fraud and Abuse Waivers - CMS,https://www.cms.gov/medicare/regulations-guida...,https://www.cms.gov/search/cms?keys=fraud&page=0
3,Trump Administration Prioritizes Affordability...,https://www.cms.gov/newsroom/press-releases/tr...,https://www.cms.gov/search/cms?keys=fraud&page=0
4,CMS Proposes Updates to Strengthen Medicare Pr...,https://www.cms.gov/newsroom/press-releases/cm...,https://www.cms.gov/search/cms?keys=fraud&page=0
...,...,...,...
85,PFS Look-up Tool Overview - CMS,https://www.cms.gov/medicare/physician-fee-sch...,https://www.cms.gov/search/cms?keys=fraud&page=9
86,Search the Physician Fee Schedule - CMS,https://www.cms.gov/medicare/physician-fee-sch...,https://www.cms.gov/search/cms?keys=fraud&page=9
87,ICD-10 - CMS,https://www.cms.gov/medicare/coding-billing/ic...,https://www.cms.gov/search/cms?keys=fraud&page=9
88,CMS-L564: Request for Employment Information,https://www.cms.gov/cms-l564-request-employmen...,https://www.cms.gov/search/cms?keys=fraud&page=9


## 3. Read CMS webpages and PDFs

CMS search results may point to HTML webpages or PDF documents. Therefore, I developed a function to extract information from those elements too. 

In [20]:
def clean_text(text):
    """Remove repeated whitespace."""
    return re.sub(r"\s+", " ", text).strip()


def read_pdf(content):
    """Extract text from PDF bytes."""
    reader = PdfReader(io.BytesIO(content))
    text = " ".join(page.extract_text() or "" for page in reader.pages)
    return clean_text(text)


def read_html(html):
    """Extract readable text from the main webpage content."""
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script", "style","nav", "header", "footer",
        "form", "button", "svg", "noscript"
    ]):
        tag.decompose()

    main = soup.find("main") or soup.find("article") or soup.body or soup
    return clean_text(main.get_text(" ", strip=True))


def read_source(item):
    """Download one CMS source and return cleaned text."""
    response = session.get(item["url"],timeout=TIMEOUT)
    response.raise_for_status()

    content_type = response.headers.get("content-type", "").lower()
    is_pdf = (
        "application/pdf" in content_type
        or item["url"].lower().endswith(".pdf")
    )

    text = read_pdf(response.content) if is_pdf else read_html(response.text)

    return {
        "title": item["title"],
        "url": item["url"],
        "type": "PDF" if is_pdf else "Webpage",
        "text": text[:MAX_SOURCE_CHARACTERS],
    }


sources = []
source_errors=[]

for number, item in enumerate(search_results, start=1):
    print(f"{number}/{len(search_results)} Reading: {item['title']}")

    try:
        sources.append(read_source(item))
    except Exception as error:
        source_errors.append({
            "title": item["title"],
            "url": item["url"],
            "error": str(error),
        })

    time.sleep(REQUEST_DELAY)

sources_df = pd.DataFrame([
    {
        "title": source["title"],
        "type": source["type"],
        "characters": len(source["text"]),
        "url": source["url"],
    }
    for source in sources
])

display(sources_df)

if source_errors:
    print("Skipped sources:")
    display(pd.DataFrame(source_errors))

if not sources:
    raise RuntimeError("No CMS sources could be read.")

1/90 Reading: About searching
2/90 Reading: CMS Announces Aggressive Nationwide Crackdown on Fraud with ...
3/90 Reading: Fraud and Abuse Waivers - CMS
4/90 Reading: Trump Administration Prioritizes Affordability by Announcing Major ...
5/90 Reading: CMS Proposes Updates to Strengthen Medicare Program Integrity ...
6/90 Reading: Crushing Fraud Chili Cook-Off Competition - CMS
7/90 Reading: Reporting Fraud - CMS
8/90 Reading: Performance Standard for Referrals of Suspected Fraud from ... - CMS
9/90 Reading: Healthcare Fraud Prevention Partnership | CMS
10/90 Reading: June 1-5 is Medicare Fraud Prevention Week. Here's How ... - CMS
11/90 Reading: Medicare Fraud & Abuse: Prevent, Detect, Report - CMS
12/90 Reading: CMS Final Rule Lowers Costs, Cracks Down on Fraud , and ...
13/90 Reading: The Health Care Fraud and Abuse Control Program Protects ... - CMS
14/90 Reading: Crushing Fraud , Waste, & Abuse | CMS
15/90 Reading: Fraud Prevention Toolkit | CMS
16/90 Reading: Common Types of Health

,title,type,characters,url
0,About searching,Webpage,615,https://www.cms.gov/search/cms/help
1,CMS Announces Aggressive Nationwide Crackdown ...,Webpage,5195,https://www.cms.gov/newsroom/press-releases/cm...
2,Fraud and Abuse Waivers - CMS,Webpage,13189,https://www.cms.gov/medicare/regulations-guida...
3,Trump Administration Prioritizes Affordability...,Webpage,9408,https://www.cms.gov/newsroom/press-releases/tr...
4,CMS Proposes Updates to Strengthen Medicare Pr...,Webpage,6255,https://www.cms.gov/newsroom/press-releases/cm...
...,...,...,...,...
85,PFS Look-up Tool Overview - CMS,Webpage,1982,https://www.cms.gov/medicare/physician-fee-sch...
86,Search the Physician Fee Schedule - CMS,Webpage,246,https://www.cms.gov/medicare/physician-fee-sch...
87,ICD-10 - CMS,Webpage,15441,https://www.cms.gov/medicare/coding-billing/ic...
88,CMS-L564: Request for Employment Information,Webpage,7206,https://www.cms.gov/cms-l564-request-employmen...


## 4. Summarize each CMS source

Each source is summarized into only fraud types and CMS detection methods.

If a source contains no relevant info, the model returns empty lists.

In [21]:
SOURCE_PROMPT = '''
Summarize the supplied official CMS source.

Use only the supplied CMS text. Do not add outside knowledge.

Return valid JSON:

{
  "fraud_items": [
    {
      "category": "broad CMS-supported fraud category",
      "fraud_type": "specific CMS-supported fraud type",
      "description": "one concise sentence",
      "red_flags": ["CMS-supported warning signs"],
      "detection_methods": ["CMS-supported detection or review methods"]
    }
  ],
  "detection_technologies": [
    {
      "technology": "CMS-described method or technology",
      "purpose": "how CMS says it helps detect fraud",
      "evidence": "brief paraphrase of the CMS source"
    }
  ]
}

Rules:
- Extract only fraud-related information.
- Keep category, fraud type, red flags, and detection separate.
- Do not include prevention or reporting guidance.
- Do not add external algorithms or model names.
- A red flag is not proof of fraud.
- Return empty lists when the source is not relevant.
- Use concise language.
- Return JSON only.
'''




def summarize_source(source):
    
    prompt = f'''
SOURCE TITLE:
{source["title"]}

SOURCE URL:
{source["url"]}

CMS TEXT:
{source["text"]}
'''

    response=client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SOURCE_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )

    summary = json.loads(response.choices[0].message.content)
    summary["source_title"] = source["title"]
    summary["source_url"] = source["url"]
    return summary


source_summaries = []

for number, source in enumerate(sources, start=1):
    print(f"{number}/{len(sources)} Summarizing: {source['title']}")
    source_summaries.append(summarize_source(source))

print("Completed source summaries:",len(source_summaries))

1/90 Summarizing: About searching
2/90 Summarizing: CMS Announces Aggressive Nationwide Crackdown on Fraud with ...
3/90 Summarizing: Fraud and Abuse Waivers - CMS
4/90 Summarizing: Trump Administration Prioritizes Affordability by Announcing Major ...
5/90 Summarizing: CMS Proposes Updates to Strengthen Medicare Program Integrity ...
6/90 Summarizing: Crushing Fraud Chili Cook-Off Competition - CMS
7/90 Summarizing: Reporting Fraud - CMS
8/90 Summarizing: Performance Standard for Referrals of Suspected Fraud from ... - CMS
9/90 Summarizing: Healthcare Fraud Prevention Partnership | CMS
10/90 Summarizing: June 1-5 is Medicare Fraud Prevention Week. Here's How ... - CMS
11/90 Summarizing: Medicare Fraud & Abuse: Prevent, Detect, Report - CMS
12/90 Summarizing: CMS Final Rule Lowers Costs, Cracks Down on Fraud , and ...
13/90 Summarizing: The Health Care Fraud and Abuse Control Program Protects ... - CMS
14/90 Summarizing: Crushing Fraud , Waste, & Abuse | CMS
15/90 Summarizing: Fraud Pr

## 5. Merge and normalize CMS info

This step combines all page-level summaries, removes duplicates, and normalizes equivalent labels.


In [22]:
MERGE_PROMPT = '''
Combine all supplied CMS source summaries into one CMS-only result.

Use only the supplied information.

Normalization rules:
- Treat Health Care Fraud, Healthcare Fraud, health care fraud,
  Heal Care Fraud, Fraud in Healthcare Programs, and similar wording
  as the same broad category.
- Use the standard label "Healthcare Fraud".
- Merge duplicates caused by capitalization, spacing, punctuation,
  singular/plural wording, or obvious spelling errors.
- Keep genuinely distinct CMS-supported fraud types separate.
- Include Medicaid Fraud only when the CMS summaries contain
  Medicaid-specific fraud information.

Return valid JSON:

{
  "fraud_categories": [
    {
      "category": "normalized broad fraud category",
      "summary": "concise CMS-supported summary",
      "fraud_types": ["specific CMS-supported fraud types"],
      "sources": ["CMS source titles"]
    }
  ],
  "fraud_types": [
    {
      "category": "normalized broad fraud category",
      "fraud_type": "specific fraud type",
      "description": "one concise sentence",
      "red_flags": ["concise CMS-supported warning signs"],
      "detection_methods": ["concise CMS-supported detection methods"],
      "sources": ["CMS source titles"]
    }
  ],
  "detection_technologies": [
    {
      "technology": "CMS-described detection method or technology",
      "purpose": "concise explanation",
      "examples": ["CMS-supported examples"],
      "sources": ["CMS source titles"]
    }
  ]
}

Requirements:
- Remove repetition.
- Use simple and concise language.
- Do not include prevention or reporting.
- Do not add external algorithms or model names.
- Return JSON only.
'''


response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": MERGE_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                source_summaries,
                ensure_ascii=False,
            ),
        },
    ],
)

final_data = json.loads(response.choices[0].message.content)

## 6. Cleaning the tables


In [23]:
def to_text(value):
    """Convert lists and scalar values into readable text."""
    if isinstance(value, list):
        return "; ".join(
            str(item).strip()
            for item in value
            if str(item).strip()
        )

    if value is None:
        return ""

    return str(value).strip()


def normalize_label(value):
    """Normalize common healthcare-fraud wording variants."""
    text = str(value or "").strip()
    key = re.sub(r"[^a-z]", "", text.lower())

    healthcare_variants = {
        "healthcarefraud",
        "healthcarefrauds",
        "healcarefraud",
        "healthcarfraud",
        "fraudinhealthcareprogram",
        "fraudinhealthcareprograms",
    }

    if key in healthcare_variants:
        return "Healthcare Fraud"

    return text.title()


def unique_text(values):
    """Merge unique text values while preserving order."""
    output = []
    seen = set()

    for value in values:
        parts = value if isinstance(value, list) else str(value or "").split(";")

        for part in parts:
            item = str(part).strip()
            key = item.lower()

            if item and key not in seen:
                output.append(item)
                seen.add(key)

    return "; ".join(output)

## 7. Fraud categories

In [24]:


def to_text(value):
    if isinstance(value,list):
        return "; ".join(str(v).strip() for v in value if str(v).strip())
    return "" if value is None else str(value).strip()

def unique_text(values):
    seen=[]
    out=[]
    for v in values:
        for p in (v if isinstance(v,list) else str(v).split(";")):
            p=str(p).strip()
            if p and p.lower() not in seen:
                seen.append(p.lower())
                out.append(p)
    return "; ".join(out)

rows=[]
for item in final_data.get("fraud_categories",[]):
    rows.append({
        "Summary": to_text(item.get("summary")),
        "Fraud types": to_text(item.get("fraud_types")),
        "CMS sources": to_text(item.get("sources"))
    })

fraud_type_summary_df=pd.DataFrame(rows)

if not fraud_type_summary_df.empty:
    fraud_type_summary_df=(fraud_type_summary_df
        .groupby("Summary",as_index=False)
        .agg({"Fraud types":unique_text,"CMS sources":unique_text})
    )

display(fraud_type_summary_df)

,Summary,Fraud types,CMS sources
0,Healthcare fraud involves various fraudulent a...,Fraud by criminals masquerading as health care...,CMS Announces Aggressive Nationwide Crackdown ...


## 8. Specific fraud types and detection methods

In [25]:
# Specific Fraud Types

def normalize_type(x):
    return "" if x is None else str(x).strip().title()

rows=[]
for item in final_data.get("fraud_types",[]):
    rows.append({
        "Fraud type": normalize_type(item.get("fraud_type")),
        "Description": to_text(item.get("description")),
        "Red flags": to_text(item.get("red_flags")),
        "Detection methods": to_text(item.get("detection_methods")),
        "CMS sources": to_text(item.get("sources"))
    })

fraud_df=pd.DataFrame(rows)

if not fraud_df.empty:
    fraud_df=(fraud_df.groupby("Fraud type",as_index=False)
              .agg({
                "Description":unique_text,
                "Red flags":unique_text,
                "Detection methods":unique_text,
                "CMS sources":unique_text
              })
              .sort_values("Fraud type"))

display(fraud_df)

,Fraud type,Description,Red flags,Detection methods,CMS sources
0,Billing For Services Not Provided,Providers submit claims for services that were...,Discrepancies between billed services and pati...,Claims audits; Verification of service deliver...,Health Care Fraud and Program Integrity: An Ov...
1,Billing For Unnecessary Services,Providers bill for services that are not medic...,Unusual billing patterns; High volume of unnec...,Auditing claims; Reviewing patient records for...,Health Care Fraud and Program Integrity: An Ov...
2,Dmepos Fraud,Fraudulent billing practices by Durable Medica...,Unusually high spending in certain service are...,Data-driven analysis of spending patterns; Rea...,Trump Administration Prioritizes Affordability...
3,False Claims,Submitting false or fraudulent claims to gover...,Billing for services not provided; Upcoding; B...,Civil legal actions under the False Claims Act...,Laws Against Health Care Fraud | Fact Sheet | CMS
4,Fraud By Criminals Masquerading As Health Care...,Criminals pose as legitimate health care provi...,Emerging or migrating fraud schemes; Suspiciou...,Medicare Fraud Strike Force; Data analysis and...,The Health Care Fraud and Abuse Control Progra...
5,Fraudulent Applications In Aca Marketplace,Fraudsters submit fraudulent applications to t...,Unauthorized enrollments; Consumer complaints ...,Strengthening complaint resolution processes; ...,"Crushing Fraud , Waste, & Abuse | CMS"
6,Fraudulent Wound Care,Defendants submitted approximately $1.1 billio...,Medically unnecessary treatments; Kickbacks fr...,Data analytics; Investigative actions by law e...,National Health Care Fraud Takedown Results in...
7,Improper Billing And Fraudulent Activity,Fraudulent activities are prevalent in the hos...,High-risk categories for fraudulent activity; ...,Targeted investigations; Advanced data analyti...,CMS Announces Aggressive Nationwide Crackdown ...
8,Kickbacks,Involves soliciting or receiving remuneration ...,Unusual financial arrangements with providers;...,Monitoring financial transactions; Investigati...,Health Care Fraud and Program Integrity: An Ov...
9,Medical Identity Theft,Involves the appropriation or misuse of a pati...,Use of identifying information without authori...,Monitoring billing and compliance processes; M...,Health Care Fraud and Program Integrity: An Ov...


## 9. CMS fraud-detection methods and technologies

In [26]:
technology_rows=[]

for item in final_data.get("detection_technologies", []):
    technology_rows.append({
        "Detection method or technology": to_text(item.get("technology")),
        "Purpose": to_text(item.get("purpose")),
        "Examples": to_text(item.get("examples")),
        "CMS sources": to_text(item.get("sources")),
    })

technology_df=pd.DataFrame(technology_rows)

if not technology_df.empty:
    technology_df = (
        technology_df
        .groupby("Detection method or technology", as_index=False)
        .agg({
            "Purpose": unique_text,
            "Examples": unique_text,
            "CMS sources": unique_text,
        })
        .sort_values("Detection method or technology")
        .reset_index(drop=True)
    )

display(technology_df)

,Detection method or technology,Purpose,Examples,CMS sources
0,Advanced data analytics,Helps identify and investigate fraudulent acti...,CMS is intensifying investigations and using d...,CMS Announces Aggressive Nationwide Crackdown ...
1,Claims Auditing,Identifies discrepancies in billing and servic...,Claims audits can reveal patterns of fraudulen...,"Partners in Integrity: Preventing Fraud , Wast..."
2,Data Analytics,Helps detect anomalous billing and fraudulent ...,The Health Care Fraud Unit’s Data Analytics Te...,National Health Care Fraud Takedown Results in...
3,Fraud Prevention System (FPS),Identifies aberrant and suspicious billing pat...,FPS applies predictive analytics to Medicare c...,The Health Care Fraud and Abuse Control Progra...
4,Medicare Data Analysis Techniques,To identify and combat Medicare fraud through ...,The Medicare Fraud Strike Force uses data anal...,MEDICARE FRAUD STRIKE FORCE CHARGES 91 INDIVID...
5,OASIS-C,To ensure accurate patient assessments and pay...,"CMS requires OASIS submission for payment, imp...",CMS1230142 | CMS
6,Predictive Modeling,Helps identify potential fraud by analyzing pa...,CMS uses predictive modeling to detect anomali...,Health Care Fraud and Program Integrity: An Ov...
7,Real-time Monitoring,Allows for immediate detection and prevention ...,CMS is leading the fight to protect Medicare a...,National Health Care Fraud Takedown Results in...


In [27]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.


## 10. Creating the final summary

In [28]:
def make_table(dataframe, hidden_columns=None):
    """Convert a DataFrame into a clean Markdown table."""
    hidden_columns = hidden_columns or []

    if dataframe.empty:
        return "No supported information was found."

    return (
        dataframe
        .drop(columns=hidden_columns, errors="ignore")
        .to_markdown(index=False)
    )


def create_summary(frauds, technologies):
    """Create a concise CMS-only Markdown report."""

    fraud_table = make_table(
        frauds,
        hidden_columns=["CMS sources"],
    )

    technology_table = make_table(
        technologies,
        hidden_columns=["CMS sources"],
    )

    source_list = "\n".join(
        f"- [{item['title']}]({item['url']})"
        for item in search_results
    )

    return f"""# CMS Fraud Types and Detection Methods

## Scope

This report summarizes CMS fraud information collected from search pages
`page=0` through `page=9`.

Only fraud types, red flags, detection methods, and CMS-described detection
technologies are included.

## Fraud Types and Detection Methods

{fraud_table}

## CMS Fraud-Detection methods and Technologies

{technology_table}

## CMS Sources Reviewed

{source_list}
"""


summary_markdown = create_summary(
    fraud_df,
    technology_df,
)

display(Markdown(summary_markdown))

# CMS Fraud Types and Detection Methods

## Scope

This report summarizes CMS fraud information collected from search pages
`page=0` through `page=9`.

Only fraud types, red flags, detection methods, and CMS-described detection
technologies are included.

## Fraud Types and Detection Methods

| Fraud type                                               | Description                                                                                                                                  | Red flags                                                                                                                                                              | Detection methods                                                                                                                                                               |
|:---------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Billing For Services Not Provided                        | Providers submit claims for services that were never delivered.                                                                              | Discrepancies between billed services and patient records; Patient complaints about services not received                                                              | Claims audits; Verification of service delivery with patients                                                                                                                   |
| Billing For Unnecessary Services                         | Providers bill for services that are not medically necessary.                                                                                | Unusual billing patterns; High volume of unnecessary treatments                                                                                                        | Auditing claims; Reviewing patient records for medical necessity                                                                                                                |
| Dmepos Fraud                                             | Fraudulent billing practices by Durable Medical Equipment, Prosthetics, Orthotics, and Supplies (DMEPOS) suppliers.                          | Unusually high spending in certain service areas; Rapid growth in DMEPOS supplier applications; Claims involving unsupported or potentially fraudulent Medicaid claims | Data-driven analysis of spending patterns; Real-time enforcement strategies; Cross-agency coordination and law enforcement partnerships                                         |
| False Claims                                             | Submitting false or fraudulent claims to government health care programs.                                                                    | Billing for services not provided; Upcoding; Billing for unnecessary services                                                                                          | Civil legal actions under the False Claims Act; Audits and reviews by government agencies                                                                                       |
| Fraud By Criminals Masquerading As Health Care Providers | Criminals pose as legitimate health care providers to commit fraud.                                                                          | Emerging or migrating fraud schemes; Suspicious billing patterns                                                                                                       | Medicare Fraud Strike Force; Data analysis and predictive analytics                                                                                                             |
| Fraudulent Applications In Aca Marketplace               | Fraudsters submit fraudulent applications to the ACA Marketplace, collecting improper commissions.                                           | Unauthorized enrollments; Consumer complaints about enrollment                                                                                                         | Strengthening complaint resolution processes; Recouping improperly paid payments                                                                                                |
| Fraudulent Wound Care                                    | Defendants submitted approximately $1.1 billion in fraudulent claims for medically unnecessary wound treatments.                             | Medically unnecessary treatments; Kickbacks from fraudulent billing                                                                                                    | Data analytics; Investigative actions by law enforcement                                                                                                                        |
| Improper Billing And Fraudulent Activity                 | Fraudulent activities are prevalent in the hospice and home health sectors, exploiting vulnerable Medicare patients.                         | High-risk categories for fraudulent activity; Suspicious billing patterns; Changes in majority ownership to obscure control                                            | Targeted investigations; Advanced data analytics; Site visits to verify operations; Heightened oversight of newly enrolled providers; Publicly available hospice scoring system |
| Kickbacks                                                | Involves soliciting or receiving remuneration for patient referrals.                                                                         | Unusual financial arrangements with providers; Incentives offered for referrals                                                                                        | Monitoring financial transactions; Investigating referral patterns                                                                                                              |
| Medical Identity Theft                                   | Involves the appropriation or misuse of a patient’s or provider’s unique medical identifying information.                                    | Use of identifying information without authorization; Billing for unnecessary or unprovided services                                                                   | Monitoring billing and compliance processes; Managing enrollment information with payers                                                                                        |
| Outlier Payment Fraud                                    | Fraudulent claims related to outlier payments by home health agencies.                                                                       | Increased case-mix not due to patient condition; Outlier payments exceeding capped limits                                                                              | Review of outlier payment claims; Analysis of case-mix data                                                                                                                     |
| Prescription Drug Misuse And Fraud                       | CMS is taking steps to identify and prevent prescription drug fraud and abuse in the Medicare Part D program.                                | Suspect claims for prescription drugs; High volume of prescriptions for opioids                                                                                        | Investigating suspect claims before payment; Prior authorization requirements for drugs susceptible to abuse; Enhanced drug utilization tools                                   |
| Prescription Opioid Trafficking                          | Defendants illegally diverted over 15 million pills of prescription opioids and other controlled substances.                                 | Unlawful distribution of large quantities of opioids; Involvement of licensed medical professionals                                                                    | Coordinated law enforcement actions; Data analysis                                                                                                                              |
| Provider Fraud                                           | Fraudulent activities by providers or beneficiaries in Medicaid managed care.                                                                | Extreme quantities of services provided on the same day; Medically impossible or unlikely services; Improperly combined or separated services                          | Analyzing claims data; Auditing suspicious activities; Encouraging reporting from employees and beneficiaries                                                                   |
| Telemedicine And Genetic Testing Fraud                   | Defendants submitted over $1.17 billion in fraudulent claims to Medicare through deceptive telemarketing and false claims for genetic tests. | Deceptive telemarketing campaigns; Fraudulent claims for unnecessary services                                                                                          | Data analytics; Real-time monitoring                                                                                                                                            |
| Transnational Criminal Organizations                     | Defendants submitted over $12 billion in fraudulent claims through a network exploiting stolen identities.                                   | Use of foreign straw owners; Exploitation of stolen identities                                                                                                         | Proactive data analytics; International cooperation in law enforcement                                                                                                          |
| Unauthorized Hospice Enrollment                          | Beneficiaries enrolled in hospice care without their consent.                                                                                | Loss of access to regular benefits; Reports of neglect or harm to beneficiaries                                                                                        | Monitoring beneficiary complaints; Investigating reports from Senior Medicare Patrol                                                                                            |

## CMS Fraud-Detection methods and Technologies

| Detection method or technology    | Purpose                                                                       | Examples                                                                                                                  |
|:----------------------------------|:------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------|
| Advanced data analytics           | Helps identify and investigate fraudulent activities in real-time.            | CMS is intensifying investigations and using data-driven methods to prevent fraud.                                        |
| Claims Auditing                   | Identifies discrepancies in billing and service delivery.                     | Claims audits can reveal patterns of fraudulent billing practices.                                                        |
| Data Analytics                    | Helps detect anomalous billing and fraudulent schemes.                        | The Health Care Fraud Unit’s Data Analytics Team used cutting-edge data analytics to identify and support investigations. |
| Fraud Prevention System (FPS)     | Identifies aberrant and suspicious billing patterns before payments are made. | FPS applies predictive analytics to Medicare claims, showing significant savings and a strong return on investment.       |
| Medicare Data Analysis Techniques | To identify and combat Medicare fraud through data analysis.                  | The Medicare Fraud Strike Force uses data analysis to flag suspicious activity and investigate fraud.                     |
| OASIS-C                           | To ensure accurate patient assessments and payment conditions.                | CMS requires OASIS submission for payment, implementing a new version to enhance data accuracy.                           |
| Predictive Modeling               | Helps identify potential fraud by analyzing patterns in claims data.          | CMS uses predictive modeling to detect anomalies in billing that may indicate fraud.                                      |
| Real-time Monitoring              | Allows for immediate detection and prevention of fraudulent claims.           | CMS is leading the fight to protect Medicare and Medicaid through advanced data analytics and real-time monitoring.       |

## CMS Sources Reviewed

- [About searching](https://www.cms.gov/search/cms/help)
- [CMS Announces Aggressive Nationwide Crackdown on Fraud with ...](https://www.cms.gov/newsroom/press-releases/cms-announces-aggressive-nationwide-crackdown-fraud-six-month-hospice-home-health-agency-enrollment)
- [Fraud and Abuse Waivers - CMS](https://www.cms.gov/medicare/regulations-guidance/physician-self-referral/fraud-and-abuse-waivers)
- [Trump Administration Prioritizes Affordability by Announcing Major ...](https://www.cms.gov/newsroom/press-releases/trump-administration-prioritizes-affordability-announcing-major-crackdown-health-care-fraud)
- [CMS Proposes Updates to Strengthen Medicare Program Integrity ...](https://www.cms.gov/newsroom/press-releases/cms-proposes-updates-strengthen-medicare-program-integrity-combat-fraud-expand-access-home-health)
- [Crushing Fraud Chili Cook-Off Competition - CMS](https://www.cms.gov/priorities/crushing-fraud-waste-abuse/overview/crushing-fraud-chili-cook-competition)
- [Reporting Fraud - CMS](https://www.cms.gov/medicare/medicaid-coordination/center-program-integrity/reporting-fraud)
- [Performance Standard for Referrals of Suspected Fraud from ... - CMS](https://www.cms.gov/medicare-medicaid-coordination/fraud-prevention/medicaid-integrity-program/education/resource-library/performance-standard-referrals-suspected-fraud-single-state-agency-medicaid-fraud-control-unit)
- [Healthcare Fraud Prevention Partnership | CMS](https://www.cms.gov/medicare/medicaid-coordination/healthcare-fraud-prevention-partnership)
- [June 1-5 is Medicare Fraud Prevention Week. Here's How ... - CMS](https://www.cms.gov/newsroom/blog/june-1-5-medicare-fraud-prevention-week-heres-how-americans-can-help-protect-themselves-medicare)
- [Medicare Fraud & Abuse: Prevent, Detect, Report - CMS](https://www.cms.gov/Outreach-and-Education/MLN/WBT/MedicareFraudandAbuse/FraudandAbuse/story.html)
- [CMS Final Rule Lowers Costs, Cracks Down on Fraud , and ...](https://www.cms.gov/newsroom/press-releases/cms-final-rule-lowers-costs-cracks-down-fraud-expands-state-control)
- [The Health Care Fraud and Abuse Control Program Protects ... - CMS](https://www.cms.gov/newsroom/fact-sheets/health-care-fraud-abuse-control-program-protects-consumers-taxpayers-combating-health-care-fraud)
- [Crushing Fraud , Waste, & Abuse | CMS](https://www.cms.gov/fraud)
- [Fraud Prevention Toolkit | CMS](https://www.cms.gov/training-education/partner-outreach-resources/partner-with-cms/fraud-prevention-toolkit)
- [Common Types of Health Care Fraud Fact Sheet - CMS](https://www.cms.gov/files/document/overviewfwacommonfraudtypesfactsheet072616pdf)
- [CMS Launches New Model to Target Wasteful, Inappropriate ...](https://www.cms.gov/newsroom/press-releases/cms-launches-new-model-target-wasteful-inappropriate-services-original-medicare)
- [HFPP: About the Partnership - CMS](https://www.cms.gov/medicare/medicaid-coordination/healthcare-fraud-prevention-partnership/about)
- [Laws Against Health Care Fraud | Fact Sheet | CMS](https://www.cms.gov/files/document/overviewfwalawsagainstfactsheet072616pdf)
- [THE OBAMA ADMINISTRATION AND EXPANDED EFFORTS TO ...](https://www.cms.gov/newsroom/fact-sheets/obama-administration-and-expanded-efforts-fight-fraud)
- [Combating Medicare Parts C & D Fraud , Waste & Abuse - CMS](https://www.cms.gov/Outreach-and-Education/MLN/WBT/MLN3995723-MLNPartsCD/FWA/story.html)
- [National Health Care Fraud Takedown Results in 324 Defendants ...](https://www.cms.gov/newsroom/press-releases/national-health-care-fraud-takedown-results-324-defendants-charged-connection-over-14-6-billion)
- [CMS Fraud Prevention System Identified or Prevented $210 Million ...](https://www.cms.gov/newsroom/press-releases/cms-fraud-prevention-system-identified-or-prevented-210-million-improper-medicare-payments-2nd-year)
- [The Medicaid Managed Care Plan's Role in Preventing, Detecting ...](https://www.cms.gov/files/document/mcpfactsheet011416pdf)
- [Fraud & abuse | CMS](https://www.cms.gov/training-education/partner-outreach-resources/fraud-abuse)
- [VA, Health and Human Services Announce Partnership to ... - CMS](https://www.cms.gov/newsroom/press-releases/va-health-and-human-services-announce-partnership-strengthen-prevention-fraud-waste-and-abuse)
- [NEW TECHNOLOGY TO HELP FIGHT MEDICARE FRAUD - CMS](https://www.cms.gov/newsroom/press-releases/new-technology-help-fight-medicare-fraud)
- [CMS STRENGTHENS EFFORTS TO FIGHT MEDICARE WASTE ...](https://www.cms.gov/newsroom/press-releases/cms-strengthens-efforts-fight-medicare-waste-fraud-and-abuse)
- [Fiscal Year 2025 Improper Payments Fact Sheet - CMS](https://www.cms.gov/newsroom/fact-sheets/fiscal-year-2025-improper-payments-fact-sheet)
- [Fiscal Year 2024 Improper Payments Fact Sheet - CMS](https://www.cms.gov/newsroom/fact-sheets/fiscal-year-2024-improper-payments-fact-sheet)
- [HHS WOULD INCREASE REWARDS FOR REPORTING FRAUD TO ...](https://www.cms.gov/newsroom/press-releases/hhs-would-increase-rewards-reporting-fraud-nearly-10-million)
- [State by State Fraud and Abuse Reporting Contacts - CMS](https://www.cms.gov/files/document/statefraudandabusecontactreport-aug2014pdf)
- [CMS Strategy to Combat Medicare Part D Prescription Drug Fraud ...](https://www.cms.gov/newsroom/fact-sheets/cms-strategy-combat-medicare-part-d-prescription-drug-fraud-and-abuse)
- [CPI Our Mission Priorities | CMS](https://www.cms.gov/medicare/medicaid-coordination/center-program-integrity)
- [MEDICARE IMPLEMENTS NEW STEPS TO PREVENT DRUG ...](https://www.cms.gov/newsroom/press-releases/medicare-implements-new-steps-prevent-drug-card-fraud)
- [Health Care Fraud and Program Integrity Resource Guide - CMS](https://www.cms.gov/files/document/overviewfwaresourceguide072616pdf)
- [Healthcare Fraud Prevention Partnership (HFPP) - CMS](https://www.cms.gov/files/document/hfpp-flyer05082017pdf)
- [MLN Web-Based Training - CMS](https://www.cms.gov/training-education/medicare-learning-networkr-mln/resources-training/mln-web-based-training)
- [The Health Care Fraud and Abuse Control Program Protects ... - CMS](https://www.cms.gov/newsroom/fact-sheets/health-care-fraud-and-abuse-control-program-protects-consumersand-taxpayers-combating-health-care)
- [MEDICARE FRAUD STRIKE FORCE CHARGES 91 INDIVIDUALS ...](https://www.cms.gov/newsroom/press-releases/medicare-fraud-strike-force-charges-91-individuals-approximately-430-million-false-billing)
- [Fraud , Waste, and Abuse Referral Guidelines for Use by Managed ...](https://www.cms.gov/files/document/mcpreferralguidelines011416pdf)
- [Partners in Integrity: Preventing Fraud , Waste, and Abuse in ... - CMS](https://www.cms.gov/files/document/wpreventingfraudwasteandabuse081315fpdf)
- [Health Care Fraud and Program Integrity: An Overview for Providers](https://www.cms.gov/files/document/healthcarefraudandpi072616pdf)
- [Blow the Whistle on Medicaid Fraud - CMS](https://www.cms.gov/files/document/fwa-provfraudcard050216pdf)
- [Preventing Fraud , Waste, and Abuse in Medicaid Home Health ...](https://www.cms.gov/files/document/wpreventingfwahhdmefs081315fpdf)
- [Medicaid Program Integrity Educational Resources - CMS](https://www.cms.gov/medicare/medicaid-coordination/states/education)
- [List of CPT/HCPCS Codes - CMS](https://www.cms.gov/medicare/regulations-guidance/physician-self-referral/list-cpt-hcpcs-codes)
- [07/12/2007 | CMS](https://www.cms.gov/medicare/fraud-and-abuse/physicianselfreferral/significant-regulatory-history-items/cms1231350)
- [2010-08-03 | CMS](https://www.cms.gov/medicare/fraud-and-abuse/physicianselfreferral/significant-regulatory-history-items/srh-cms%253f1504%253fp)
- [2018-03-30 | CMS](https://www.cms.gov/medicare-medicaid-coordination/fraud-prevention/fraudabuseforprofs/program-integrity-review-reports-list-items/msfy17_dl)
- [08/19/2008 | CMS](https://www.cms.gov/medicare/fraud-and-abuse/physicianselfreferral/significant-regulatory-history-items/cms1231345)
- [2018-01-03 | CMS](https://www.cms.gov/medicare-medicaid-coordination/fraud-prevention/fraudabuseforprofs/program-integrity-review-reports-list-items/flfy17_dl)
- [2018-11-01 | CMS](https://www.cms.gov/medicare-medicaid-coordination/fraud-prevention/fraudabuseforprofs/program-integrity-review-reports-list-items/orfy18_dl)
- [Hospice - Key Message and Tips for Beneficiaries | CMS](https://www.cms.gov/Medicare-Medicaid-Coordination/Fraud-Prevention/Medicaid-Integrity-Program/Education/Resource-Library/hospice-key-message-and-tips-beneficiaries)
- [Personal Care and Support Services - Key Message and Tips ... - CMS](https://www.cms.gov/medicare-medicaid-coordination/fraud-prevention/medicaid-integrity-program/education/resource-library/personal-care-and-support-services-key-message-and-tips-providers)
- [CMS ANNOUNCES MEDICARE IMPROPER PAYMENTS RATE ...](https://www.cms.gov/newsroom/press-releases/cms-announces-medicare-improper-payments-rate-2003)
- [HHS OIG Work Plan for FY 2014](https://www.cms.gov/files/document/oig-work-plan-2014pdf)
- [Clinical Laboratory Improvement Amendments (CLIA) - CMS](https://www.cms.gov/medicare/quality/clinical-laboratory-improvement-amendments)
- [COVID-19 Medicaid and CHIP Data Snapshot through May 2021](https://www.cms.gov/newsroom/news-alert/covid-19-medicaid-and-chip-data-snapshot-through-may-2021)
- [Centers for Medicare & Medicaid Services (CMS) Administrator ...](https://www.cms.gov/newsroom/press-releases/centers-medicare-medicaid-services-cms-administrator-seema-verma-statement-enforcement-letter-idaho)
- [Medicaid.gov: The Official U.S. Government Site for Medicaid and ...](https://www.cms.gov/home/medicaid.asp)
- [CMS1230142 | CMS](https://www.cms.gov/medicare/medicare-fee-for-service-payment/homehealthpps/home-health-prospective-payment-system-regulations-and-notices-items/cms1230142)
- [Become a Medicare Provider or Supplier - CMS](https://www.cms.gov/medicare/enrollment-renewal/providers-suppliers)
- [AB-00-50.60 | CMS](https://www.cms.gov/regulations-and-guidance/guidance/transmittals/cms-program-memoranda-items/cms051913)
- [AB-00-23.60 | CMS](https://www.cms.gov/regulations-and-guidance/guidance/transmittals/cms-program-memoranda-items/cms051993)
- [Trump Administration Announces Medicaid and CHIP Managed ...](https://www.cms.gov/newsroom/press-releases/trump-administration-announces-medicaid-chip-managed-care-final-rule-continues-commitment-transform)
- [AB-01-31 | CMS](https://www.cms.gov/regulations-and-guidance/guidance/transmittals/cms-program-memoranda-items/cms026312)
- [MLN Connects Newsletter for July 23, 2026 - CMS](https://www.cms.gov/training-education/medicare-learning-network/newsletter/mln-connects-newsletter-july-23-2026)
- [DOCTORS, HOSPITALS PARTNER TO COORDINATE CARE ... - CMS](https://www.cms.gov/newsroom/press-releases/more-doctors-hospitals-partner-coordinate-care-people-medicare)
- [No Surprise Billing | CMS](https://www.cms.gov/nosurprises)
- [Amending the Indirect Hold Harmless Threshold of Health ... - CMS](https://www.cms.gov/newsroom/fact-sheets/amending-indirect-hold-harmless-threshold-health-care-related-taxes-proposed-rule-cms-2452-p)
- [AB-00-56.60 | CMS](https://www.cms.gov/regulations-and-guidance/guidance/transmittals/cms-program-memoranda-items/cms051919)
- [Newsroom Homepage - CMS](https://www.cms.gov/about-cms/contact/newsroom)
- [Information for Providers - CMS](https://www.cms.gov/medicare/coverage/prescription-drug-coverage/medicare-glp-1-bridge/information-providers)
- [AB-02-106 | CMS](https://www.cms.gov/regulations-and-guidance/guidance/transmittals/cms-program-memoranda-items/cms024946)
- [OBAMA ADMINISTRATION ISSUES NEW RULES TO CUT RED ...](https://www.cms.gov/newsroom/press-releases/obama-administration-issues-new-rules-cut-red-tape-doctors-and-hospitals-saving-up-9-billion)
- [AFFORDABLE CARE ACT GIVES PROVIDERS NEW OPTIONS TO ...](https://www.cms.gov/newsroom/press-releases/affordable-care-act-gives-providers-new-options-better-coordinate-health-care)
- [Centers for Medicare and Medicaid Services Releases Reports on ...](https://www.cms.gov/newsroom/press-releases/centers-medicare-medicaid-services-releases-reports-performance-exchanges-individual-health)
- [CMS awards $67 million in Affordable Care Act funding to help ...](https://www.cms.gov/newsroom/press-releases/cms-awards-67-million-affordable-care-act-funding-help-consumers-sign-up-affordable-health-insurance)
- [CMS Approves Pennsylvania's State Relief and Empowerment Waiver](https://www.cms.gov/newsroom/press-releases/cms-approves-pennsylvanias-state-relief-empowerment-waiver)
- [Kentucky, Maine, and New Mexico Launch State Marketplaces for ...](https://www.cms.gov/newsroom/press-releases/kentucky-maine-new-mexico-launch-state-marketplaces-2022-coverage)
- [HETS EDI: How to Enroll - CMS](https://www.cms.gov/data-research/cms-information-technology/hipaa-eligibility-transaction-system-hets/hets-edi-how-enroll)
- [Medicare Part B Drug Payment Limit File Page - CMS](https://www.cms.gov/medicare/payment/part-b-drugs/asp-pricing-files)
- [Medicare GLP-1 Bridge - CMS](https://www.cms.gov/medicare/coverage/prescription-drug-coverage/medicare-glp-1-bridge)
- [CMS Modernizes Nursing Home Oversight with New Risk-Based ...](https://www.cms.gov/newsroom/press-releases/cms-modernizes-nursing-home-oversight-new-risk-based-survey-approach-designed-highlight-high)
- [PFS Look-up Tool Overview - CMS](https://www.cms.gov/medicare/physician-fee-schedule/search/overview)
- [Search the Physician Fee Schedule - CMS](https://www.cms.gov/medicare/physician-fee-schedule/search)
- [ICD-10 - CMS](https://www.cms.gov/medicare/coding-billing/icd-10-codes)
- [CMS-L564: Request for Employment Information](https://www.cms.gov/cms-l564-request-employment-information)
- [Medicare Secondary Payer Recovery Portal - CMS](https://www.cms.gov/medicare/coordination-benefits-recovery/overview/secondary-payer-recovery-portal)


## 11. Save outputs

In [29]:
OUTPUT_DIR = Path("cms_fraud_pages_0_to_9_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / "CMS_Fraud_Categories_and_Detection.md"
categories_path = OUTPUT_DIR / "CMS_Fraud_Categories.csv"
fraud_types_path = OUTPUT_DIR / "CMS_Fraud_Types_and_Detection.csv"
technologies_path = OUTPUT_DIR / "CMS_Detection_Technologies.csv"
source_summaries_path = OUTPUT_DIR / "CMS_Source_Summaries.json"
search_results_path = OUTPUT_DIR / "CMS_Search_Results_Pages_0_to_9.csv"

summary_path.write_text(summary_markdown, encoding="utf-8")
fraud_type_summary_df.to_csv(categories_path, index=False)
fraud_df.to_csv(fraud_types_path, index=False)
technology_df.to_csv(technologies_path, index=False)
results_df.to_csv(search_results_path, index=False)

source_summaries_path.write_text(
    json.dumps(
        source_summaries,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved files:")
print("-", summary_path.resolve())
print("-", categories_path.resolve())
print("-", fraud_types_path.resolve())
print("-", technologies_path.resolve())
print("-", source_summaries_path.resolve())
print("-", search_results_path.resolve())

Saved files:
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Fraud_Categories_and_Detection.md
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Fraud_Categories.csv
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Fraud_Types_and_Detection.csv
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Detection_Technologies.csv
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Source_Summaries.json
- /Users/guo/Desktop/Healthcare AI Fraud_Web_Final/cms_fraud_pages_0_to_9_output/CMS_Search_Results_Pages_0_to_9.csv


## Final workflow

```text
Search CMS pages 0–9 → pull unique links → fetch pages & PDFs
→ summarize each source → sort into fraud categories
→ pull out red flags & how CMS catches them
→ write it all up as Markdown, CSV, and JSON
```